# CRISPR Off-Target Activity: Score Prediction & Mismatch Analysis

Cleaned and reorganized version. Sections:
1. Setup
2. Baseline regression (Score)
3. Baseline classification (Identity: ON vs OFF)
4. Mismatch position feature engineering (fixes a length-mismatch bug in the original)
5. CHANGE-seq deep dive
6. Score model v2 (naive split)
7. Score model v3 (grouped split by guide — the honest evaluation)

## 0. Setup

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, r2_score,
    classification_report, roc_auc_score, confusion_matrix,
)

df = pd.read_csv("allframe_update_addEpige.txt", sep="\t", low_memory=False)
print("Data shape:", df.shape)

## 1. Baseline regression: predicting Score

In [ ]:
df_reg = df.dropna(subset=["Score"]).copy()
df_reg["Score_log"] = np.log1p(df_reg["Score"])

numeric_cols = ["start", "end", "Mismach", "Mismach2", "Indel_control%"]
categorical_cols = ["Technology", "Source", "Cas9_type", "Assembly", "PAM"]
numeric_cols = [c for c in numeric_cols if c in df_reg.columns]
categorical_cols = [c for c in categorical_cols if c in df_reg.columns]

for col in numeric_cols:
    df_reg[col] = pd.to_numeric(df_reg[col], errors="coerce")
df_reg = df_reg.dropna(subset=numeric_cols)

X = df_reg[numeric_cols + categorical_cols]
y = df_reg["Score_log"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
])

reg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42)),
])

reg_pipeline.fit(X_train, y_train)
y_pred = reg_pipeline.predict(X_test)

print(f"Test MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Test R2: {r2_score(y_test, y_pred):.2f}")

rf = reg_pipeline.named_steps["regressor"]
feature_names = numeric_cols + list(
    reg_pipeline.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_cols)
)
reg_importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("\nTop 10 Feature Importances:")
print(reg_importances.head(10))

**Caveat:** `Technology` dominates feature importance here mostly because different assays report `Score` on very different numeric scales, not because it's biologically informative. Treat this as a sanity-check baseline, not a final model.

In [ ]:
comparison = pd.DataFrame({"Actual Score": y_test.values[:10], "Predicted Score": y_pred[:10]})
print(comparison)
print(y.describe())

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.4)
plt.xlabel("Actual Log(Score)")
plt.ylabel("Predicted Log(Score)")
plt.title("Predicted vs Actual Log(Score)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.show()

residuals = y_test - y_pred
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=residuals, alpha=0.4)
plt.axhline(0, color="r", linestyle="--")
plt.xlabel("Actual Log(Score)")
plt.ylabel("Residuals")
plt.title("Residuals Plot")
plt.show()

plt.figure(figsize=(10, 6))
reg_importances.head(10).sort_values().plot(kind="barh", color="skyblue")
plt.xlabel("Feature Importance")
plt.title("Top 10 Features (Regression)")
plt.show()

## 2. Baseline classification: ON vs OFF target

In [ ]:
df_clf = df.dropna(subset=["Identity"]).copy()
df_clf = df_clf[df_clf["Identity"].isin(["ON", "OFF"])]
y_clf = (df_clf["Identity"] == "ON").astype(int)

print("Class balance:")
print(df_clf["Identity"].value_counts(normalize=True))

def gc_content(seq):
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    seq = seq.upper()
    return (seq.count("G") + seq.count("C")) / len(seq)

df_clf["protospacer_gc"] = df_clf["Protospacer_sequence"].apply(gc_content)
df_clf["target_gc"] = df_clf["Target_sequence"].apply(gc_content)
df_clf["seq_length"] = df_clf["Protospacer_sequence"].apply(lambda s: len(s) if isinstance(s, str) else np.nan)

numeric_cols = ["Mismach", "Mismach2", "protospacer_gc", "target_gc", "seq_length"]
categorical_cols = ["Technology", "Cas9_type", "PAM", "Strand"]
numeric_cols = [c for c in numeric_cols if c in df_clf.columns]
categorical_cols = [c for c in categorical_cols if c in df_clf.columns]

for col in numeric_cols:
    df_clf[col] = pd.to_numeric(df_clf[col], errors="coerce")

df_clf_model = df_clf.dropna(subset=numeric_cols)
y_clf = y_clf.loc[df_clf_model.index]

X_clf = df_clf_model[numeric_cols + categorical_cols]
print(f"\nFinal modeling set: {X_clf.shape[0]} rows")
print(f"ON-target count: {y_clf.sum()}, OFF-target count: {(y_clf == 0).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

clf_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
])

clf_pipeline = Pipeline([
    ("preprocessor", clf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight="balanced", n_jobs=-1, random_state=42
    )),
])

clf_pipeline.fit(X_train, y_train)
y_pred = clf_pipeline.predict(X_test)
y_proba = clf_pipeline.predict_proba(X_test)[:, 1]

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["OFF", "ON"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

clf = clf_pipeline.named_steps["classifier"]
clf_feature_names = numeric_cols + list(
    clf_pipeline.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_cols)
)
clf_importances = pd.Series(clf.feature_importances_, index=clf_feature_names).sort_values(ascending=False)
print("\nTop 15 Feature Importances:")
print(clf_importances.head(15))

**Caveat:** `Mismach`/`Mismach2` dominate this classifier by far, and ON-target rows are essentially defined by having (near) zero mismatches. The near-perfect ROC-AUC mostly reflects that definition, not a hard classification problem — treat it as a sanity check, not evidence of a strong model.

## 3. Mismatch position feature engineering

**Bug in the original notebook:** it compared `Protospacer_sequence` to `Target_sequence` position-by-position, but `Target_sequence` includes the PAM (so the two strings are different lengths) — every row failed the length check and the computed mismatch count was silently `None` for 100% of rows (0% agreement). `Sequence_show` marks mismatched bases with lowercase letters and is the right column to use instead. There was also a reference to an undefined `df_clean` variable, which has been replaced with `df_off` below.

In [ ]:
df_off = df[df["Identity"] == "OFF"].copy()
df_off["Mismach_numeric"] = pd.to_numeric(df_off["Mismach"], errors="coerce")

def mismatch_positions_from_display(seq_show):
    """Sequence_show marks mismatched bases with lowercase letters."""
    if not isinstance(seq_show, str):
        return None
    return [i for i, ch in enumerate(seq_show) if ch.islower()]

df_off["mismatch_positions"] = df_off["Sequence_show"].apply(mismatch_positions_from_display)
df_off["computed_mismatch_count"] = df_off["mismatch_positions"].apply(
    lambda x: len(x) if x is not None else np.nan
)

df_off["agree"] = df_off["computed_mismatch_count"] == df_off["Mismach_numeric"]
agreement = df_off["agree"].mean()
print(f"Agreement between computed and reported mismatch count: {agreement:.2%}")

In [ ]:
# Agreement and PAM length by Cas9 type
print(df_off.groupby("Cas9_type")["agree"].mean())
print(df_off.groupby("Cas9_type")["PAM"].apply(lambda x: x.dropna().unique()[:5]))

In [ ]:
def split_mismatches(positions, protospacer_len=20):
    if positions is None:
        return pd.Series([None, None])
    proto = [p for p in positions if p < protospacer_len]
    pam = [p for p in positions if p >= protospacer_len]
    return pd.Series([proto, pam])

df_off[["proto_mismatch_pos", "pam_mismatch_pos"]] = df_off["mismatch_positions"].apply(split_mismatches)
df_off["n_proto_mismatch"] = df_off["proto_mismatch_pos"].apply(len)
df_off["n_pam_mismatch"] = df_off["pam_mismatch_pos"].apply(len)

df_final = df_off[df_off["computed_mismatch_count"] == df_off["Mismach_numeric"]].copy()
print("Final clean rows:", df_final.shape[0])
print(df_final[["n_proto_mismatch", "n_pam_mismatch"]].describe())

In [ ]:
def min_distance_to_pam(proto_positions, protospacer_len=20):
    """Distance from the closest protospacer mismatch to the PAM-adjacent end."""
    if not proto_positions:
        return np.nan
    return min(protospacer_len - 1 - p for p in proto_positions)

df_final["min_dist_to_pam"] = df_final["proto_mismatch_pos"].apply(min_distance_to_pam)
print(df_final["min_dist_to_pam"].describe())
print(df_final["n_pam_mismatch"].value_counts())

## 4. CHANGE-seq deep dive

In [ ]:
df_final["Indel_treatment_numeric"] = pd.to_numeric(df_final["Indel_treatment%"], errors="coerce")
print(df_final["Indel_treatment_numeric"].describe())
print("Missing:", df_final["Indel_treatment_numeric"].isna().sum(), "/", len(df_final))
print(df_final.groupby("Technology")["Indel_treatment_numeric"].describe())

df_change = df_final[df_final["Technology"] == "CHANGE-seq"].copy()
print("\nCHANGE-seq rows:", df_change.shape[0])

df_change["Score_numeric"] = pd.to_numeric(df_change["Score"], errors="coerce")
df_change["protospacer_gc"] = df_change["Protospacer_sequence"].apply(gc_content)  # reuses gc_content() from Section 2

print(df_change["Score_numeric"].describe())
print(df_change[["min_dist_to_pam", "n_proto_mismatch", "n_pam_mismatch", "protospacer_gc", "Score_numeric"]].describe())

## 5. Score model v2: mismatch-position features (naive split)

In [ ]:
df_change["Score_log"] = np.log1p(df_change["Score_numeric"])
feature_cols = ["min_dist_to_pam", "n_proto_mismatch", "n_pam_mismatch", "protospacer_gc"]
df_change_model = df_change.dropna(subset=feature_cols + ["Score_log"])

X = df_change_model[feature_cols]
y = df_change_model["Score_log"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

rf = RandomForestRegressor(n_estimators=300, max_depth=12, n_jobs=-1, random_state=42)
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)

gb = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
gb.fit(X_train_s, y_train)
gb_pred = gb.predict(X_test_s)

print("Random Forest -- MSE:", mean_squared_error(y_test, rf_pred), "R2:", r2_score(y_test, rf_pred))
print("Gradient Boosting -- MSE:", mean_squared_error(y_test, gb_pred), "R2:", r2_score(y_test, gb_pred))

print("\nRF Feature importances:")
print(pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False))
print("\nGB Feature importances:")
print(pd.Series(gb.feature_importances_, index=feature_cols).sort_values(ascending=False))

df_change_model.groupby("min_dist_to_pam")["Score_log"].mean().plot(kind="bar", figsize=(10, 6))
plt.xlabel("Distance from PAM (mismatch position)")
plt.ylabel("Mean log(Score)")
plt.title("Off-target activity vs mismatch distance from PAM")
plt.show()

## 6. Guide-level check: does the naive split leak?

In [ ]:
print("Total rows:", df_change.shape[0])
print("Unique guides (Guide_sequence):", df_change["Guide_sequence"].nunique())
print("Unique on-target sites (On_target_site):", df_change["On_target_site"].nunique())

sites_per_guide = df_change.groupby("Guide_sequence").size()
print(sites_per_guide.describe())

## 7. Score model v3: grouped split by guide (the honest evaluation)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_change_model["Guide_sequence"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_guides = set(df_change_model.iloc[train_idx]["Guide_sequence"])
test_guides = set(df_change_model.iloc[test_idx]["Guide_sequence"])
print("Overlap:", len(train_guides & test_guides))  # must be 0

rf2 = RandomForestRegressor(n_estimators=300, max_depth=12, n_jobs=-1, random_state=42)
rf2.fit(X_train, y_train)
rf2_pred = rf2.predict(X_test)

gb2 = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
gb2.fit(X_train, y_train)
gb2_pred = gb2.predict(X_test)

print("Grouped split -- RF R2:", r2_score(y_test, rf2_pred), "MSE:", mean_squared_error(y_test, rf2_pred))
print("Grouped split -- GB R2:", r2_score(y_test, gb2_pred), "MSE:", mean_squared_error(y_test, gb2_pred))

**Takeaway:** the naive random split (R² ≈ 0.30) looked promising but leaked information across rows that share the same guide. The grouped split (R² ≈ 0.02–0.03) shows the model barely generalizes to off-target sites from guides it hasn't seen before — the four positional/GC features alone aren't enough to predict CHANGE-seq score for new guides.